# Custom model 

This custom model is created to understand what handcart should look like. the main objective of this model is to understand how does a cart look like to be use

Dataset is taken from roboflow:
- plane trolley:  https://universe.roboflow.com/kss/trolley-h0c3d/dataset/5#
- handcart: https://universe.roboflow.com/jhedai-things/trolley-unyeh 
- **NEW**: https://universe.roboflow.com/marwadi-university-qr1fp/trolley-detector

In [4]:
!pip install roboflow ultralytics wandb python-dotenv --quiet

In [ ]:
from roboflow import Roboflow
import os, dotenv
dotenv.load_dotenv()

rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
project = rf.workspace("kss").project("trolley-h0c3d")
version = project.version(5)
dataset = version.download("yolov8")

project = rf.workspace("marwadi-university-qr1fp").project("trolley-detector")
version = project.version(2)
dataset = version.download("yolov8")
                
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Trolley-Detector-2 in yolov8:: 100%|██████████| 11007/11007 [00:18<00:00, 600.75it/s] 


In [17]:
#  NEW: this is the step where we already combined all the dataset from the 2 dataset that we install above and relabelled them all as 1 class, which is "handcart"

data_location = r"C:\Users\jessl\repos\yolo-handcart-detection\src\data\combined_trolley_dataset"

In [18]:
from dotenv import load_dotenv
import os
import wandb

load_dotenv()

wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [19]:
import os
import wandb
from ultralytics import YOLO

# Start W&B run
run = wandb.init(
    entity=os.getenv("WANDB_ENTITY"),
    project=os.getenv("WANDB_PROJECT_NAME"),
    name="yolo_combined_trolley_dataset",
    config={
        "model": "yolov8s",
        "epochs": 10,
        "imgsz": 640,
        "batch": 16,
        "optimizer": "AdamW",
        "lr0": 0.001,
        "weight_decay": 0.0005,
        "patience": 5,
    },
)

# Load pretrained model
model = YOLO("yolov8s.pt")

# Train
results = model.train(
    data=os.path.join(data_location, "data.yaml"),
    epochs=10,
    imgsz=640,
    batch=16,

    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    patience=5,

    name="yolo_combined_trolley_dataset",
    plots=True,
)

# Evaluate
metrics = model.val(data=os.path.join(data_location, "data.yaml"))

# Log final metrics
wandb.log({
    "final/mAP50": metrics.box.map50,
    "final/mAP50-95": metrics.box.map,
    "final/precision": metrics.box.mp,
    "final/recall": metrics.box.mr,
})

run.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


New https://pypi.org/project/ultralytics/8.4.72 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.70  Python-3.13.14 torch-2.12.1+cpu CPU (12th Gen Intel Core i7-12700H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\jessl\repos\yolo-handcart-detection\src\data\combined_trolley_dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=

final/mAP50,▁
final/mAP50-95,▁
final/precision,▁
final/recall,▁
final/mAP50,0.86
final/mAP50-95,0.60654
final/precision,0.80248
final/recall,0.77931


# Evaluate the trained model

In [ ]:
# data_location = r"C:\Users\jessl\repos\yolo-handcart-detection\src\data\trolley-5"

In [20]:
from ultralytics import YOLO
import os

# load best trained model
model = YOLO(r"C:\Users\jessl\repos\yolo-handcart-detection\src\data_pipeline\runs\detect\yolo_combined_trolley_dataset-2\weights\best.pt")

# run validation
metrics = model.val(data=data_location + '\data.yaml')

print("\n===== EVALUATION METRICS =====")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

Ultralytics 8.4.70  Python-3.13.14 torch-2.12.1+cpu CPU (12th Gen Intel Core i7-12700H)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs


<>:8: SyntaxWarning: invalid escape sequence '\d'
<>:8: SyntaxWarning: invalid escape sequence '\d'
C:\Users\jessl\AppData\Local\Temp\ipykernel_23816\2528455410.py:8: SyntaxWarning: invalid escape sequence '\d'
  metrics = model.val(data=data_location + '\data.yaml')


val: Fast image access  (ping: 0.10.0 ms, read: 42.212.9 MB/s, size: 34.0 KB)
val: Scanning C:\Users\jessl\repos\yolo-handcart-detection\src\data\combined_trolley_dataset\valid\labels.cache... 1126 images, 6 backgrounds, 882 corrupt: 100% ━━━━━━━━━━━━ 1126/1126 205.3Mit/s 0.0s
val: C:\Users\jessl\repos\yolo-handcart-detection\src\data\combined_trolley_dataset\valid\images\0-3-_jpg.rf.523ed2fadacd7c74b150a7bf0f1e0fc6.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: C:\Users\jessl\repos\yolo-handcart-detection\src\data\combined_trolley_dataset\valid\images\0-3-_jpg.rf.e495c69ace1086785cdb9a06dbb7a1a2.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: C:\Users\jessl\repos\yolo-handcart-detection\src\data\combined_trolley_dataset\valid\images\0-3-_jpg.rf.f1a13dddbb960de91089a53abc48b732.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count

In [ ]:
# visualize predictions on validation set
results = model.predict(
    source=os.path.join(data_location, "/valid/images"),
    conf=0.25,
    save=True,
    save_txt=True  
)

In [ ]:
results = model.predict(
    source=os.path.join(data_location, "test/images"),
    conf=0.25,
    save=True
)

In [ ]:
result = results[0]

for box in result.boxes:
    cls = int(box.cls[0])
    conf = float(box.conf[0])
    xyxy = box.xyxy[0].tolist()

    print(f"class={cls}, conf={conf:.2f}, box={xyxy}")